In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.edge.options import Options  # Changed from chrome.options to edge.options
from datetime import datetime
import random
import json

In [9]:
auth_token = '719253dd7f364d5a8da57efbc9d091f68a2edc49'  # get it from developer tools
headless=False # want to run in the background
trending_url = "https://x.com/i/trending/2058435504362905716"  #  which post to scrape
max_scrolls = 25  # how many scrolls to do

In [10]:
class Xcraper:
    def __init__(self, auth_token, headless=False):
        self.auth_token = auth_token
        self.driver = None
        self.setup_driver(headless)
    
    def setup_driver(self, headless):
        """Configure Edge with anti-detection measures"""
        edge_options = Options()  # Changed from chrome_options to edge_options
        
        # Anti-detection settings
        edge_options.add_argument("--disable-blink-features=AutomationControlled")
        edge_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        edge_options.add_experimental_option('useAutomationExtension', False)
        
        # Random user agent
        edge_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36 Edg/120.0.0.0")
        
        # Disable notifications and other popups
        edge_options.add_argument("--disable-notifications")
        edge_options.add_argument("--disable-popup-blocking")
        
        if headless:
            edge_options.add_argument("--headless")  # Edge supports headless mode
        
        # Changed from webdriver.Chrome to webdriver.Edge
        self.driver = webdriver.Edge(options=edge_options)
        self.driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    def inject_auth_cookie(self):
        """Inject auth_token to bypass login"""
        print("🌐 Opening X.com...")
        self.driver.get("https://x.com")
        time.sleep(3)
        
        print("🍪 Injecting auth token...")
        self.driver.add_cookie({
            'name': 'auth_token',
            'value': self.auth_token,
            'domain': '.x.com',
            'path': '/',
            'secure': True,
            'httpOnly': True
        })
        
        print("🔄 Refreshing page...")
        self.driver.refresh()
        time.sleep(5)
        
        # Verify login worked
        current_url = self.driver.current_url
        if "login" not in current_url.lower():
            print("✅ Successfully logged in with auth token!")
            return True
        else:
            print("❌ Login failed. Check your auth_token.")
            return False
    
    def scrape_trending_page(self, trending_url, max_scrolls=5):
        """Scrape tweets from a specific trending page"""
        print(f"📊 Navigating to trending page...")
        self.driver.get(trending_url)
        time.sleep(5)
        
        tweets_data = []
        seen_tweet_ids = set()
        
        for scroll_num in range(max_scrolls):
            print(f"📜 Scrolling {scroll_num + 1}/{max_scrolls}...")
            
            # Find all tweet articles
            articles = self.driver.find_elements(By.CSS_SELECTOR, "article[data-testid='tweet']")
            print(f"   Found {len(articles)} tweets so far...")
            
            # Extract each tweet
            for article in articles:
                try:
                    tweet = self.extract_tweet_data(article)
                    if tweet:
                        # Create unique ID for the tweet to avoid duplicates
                        tweet_id = f"{tweet['handle']}_{tweet['text'][:50]}"
                        if tweet_id not in seen_tweet_ids:
                            seen_tweet_ids.add(tweet_id)
                            tweets_data.append(tweet)
                            print(f"   ✅ Collected: @{tweet['handle']}")
                except Exception as e:
                    continue
            
            # Scroll down for more content
            if scroll_num < max_scrolls - 1:
                self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(random.uniform(3, 5))
        
        print(f"\n📊 Total unique tweets collected: {len(tweets_data)}")
        #close driver
        self.driver.quit()
        return tweets_data
    
    def extract_tweet_data(self, article):
        """Extract all relevant data from a tweet article"""
        try:
            # Extract username and handle
            name = "N/A"
            handle = "N/A"
            
            try:
                username_elem = article.find_element(By.CSS_SELECTOR, "div[data-testid='User-Name']")
                links = username_elem.find_elements(By.TAG_NAME, "a")
                if len(links) >= 2:
                    name = links[0].text if links[0].text else "N/A"
                    handle = links[1].text if links[1].text else "N/A"
            except:
                pass
            
            # Extract tweet text
            tweet_text = ""
            try:
                text_elem = article.find_elements(By.CSS_SELECTOR, "div[data-testid='tweetText']")
                if text_elem:
                    tweet_text = text_elem[0].text
            except:
                pass
            
            # Extract timestamp
            timestamp = None
            try:
                time_elem = article.find_element(By.TAG_NAME, "time")
                timestamp = time_elem.get_attribute("datetime")
            except:
                pass
            
            # Extract engagement metrics
            return {
                'name': name,
                'handle': handle,
                'text': tweet_text,
                'timestamp': timestamp,
                'collected_at': datetime.now().isoformat()
            }
            
        except Exception as e:
            return None
            
    def close(self):
        if self.driver:
            self.driver.quit()
            print("🔒 Browser closed.")

In [11]:
scraper = Xcraper(auth_token = auth_token, headless=False )

In [12]:
# Inject cookie and login
if scraper.inject_auth_cookie():
    # Scrape the trending page
    print("\n📊 Starting to scrape tweets...")
    tweets = scraper.scrape_trending_page(trending_url = trending_url, max_scrolls = max_scrolls)

🌐 Opening X.com...
🍪 Injecting auth token...
🔄 Refreshing page...
✅ Successfully logged in with auth token!

📊 Starting to scrape tweets...
📊 Navigating to trending page...
📜 Scrolling 1/25...
   Found 3 tweets so far...
   ✅ Collected: @@PiyushGoyal
   ✅ Collected: @@IndiaAllliance
   ✅ Collected: @@sambitswaraj
📜 Scrolling 2/25...
   Found 10 tweets so far...
   ✅ Collected: @@sathyashrii
   ✅ Collected: @@princepoptimus
   ✅ Collected: @@Shekharcoool5
   ✅ Collected: @@TheAliceSmith
   ✅ Collected: @@pradip103
   ✅ Collected: @@NewsArenaIndia
   ✅ Collected: @@Sarcaztick
   ✅ Collected: @@apnarajeevnigam
   ✅ Collected: @@TS_SinghDeo
   ✅ Collected: @@AjayLalluINC
📜 Scrolling 3/25...
   Found 7 tweets so far...
   ✅ Collected: @@MaktoobMedia
   ✅ Collected: @@RuhiTewari
   ✅ Collected: @@TheChronology__
   ✅ Collected: @@iramsubramanian
   ✅ Collected: @@RiaRevealed
   ✅ Collected: @@shaandelhite
   ✅ Collected: @@OpIndia_com
📜 Scrolling 4/25...
   Found 7 tweets so far...
   ✅ Coll

In [13]:
if tweets:
    # Save to CSV
    df = pd.DataFrame(tweets)
    csv_filename = f"Twitter Data {datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    df.to_csv(csv_filename, index=False, encoding='utf-8')
    print(f"\n💾 Saved {len(tweets)} tweets to {csv_filename}")


💾 Saved 145 tweets to Twitter Data 20260525_103255.csv


In [18]:
df.describe(include = 'all')

,name,handle,text,timestamp,collected_at
count,145,145,145,145,145
unique,135,135,144,145,145
top,OpIndia.com,@OpIndia_com,The Modi government will fall within 1 year\n\...,2026-05-24T04:42:31.000Z,2026-05-25T10:31:04.140113
freq,2,2,2,1,1


In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   name          145 non-null    str  
 1   handle        145 non-null    str  
 2   text          145 non-null    str  
 3   timestamp     145 non-null    str  
 4   collected_at  145 non-null    str  
dtypes: str(5)
memory usage: 5.8 KB


# EDA

In [25]:
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
from datetime import datetime
from collections import Counter
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# For sentiment analysis
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')

In [27]:
# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_colwidth', 100)

In [28]:
# data load
df = pd.read_csv('Twitter Data 20260525_103255.csv')

In [29]:
print("="*60)
print("INITIAL DATA INSPECTION")
print("="*60)

print(f"\nDataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nFirst 5 rows:\n{df.head()}")
print(f"\nMissing Values:\n{df.isnull().sum()}")

# Basic info
print("\n" + "="*60)
print("DATA INFO")
print("="*60)
df.info()

INITIAL DATA INSPECTION

Dataset Shape: (145, 5)

Columns: ['name', 'handle', 'text', 'timestamp', 'collected_at']

Data Types:
name            str
handle          str
text            str
timestamp       str
collected_at    str
dtype: object

First 5 rows:
                    name           handle  \
0           Piyush Goyal     @PiyushGoyal   
1           India गठबंधन  @IndiaAllliance   
2           Sambit Patra    @sambitswaraj   
3              SriSathya     @sathyashrii   
4  Loustinianós Augustus  @princepoptimus   

                                                                                                  text  \
0  Rahul Gandhi's statement points towards a major conspiracy by the Congress, along with all oppos...   
1  The Modi government will fall within 1 year\n\n- LOP Rahul Gandhi\n\nFor your information, let m...   
2  Rahul Gandhi said in the Congress Party's minority advisory meeting that, "In the next one year,...   
3                                               

In [30]:
duplicates = df.duplicated(subset=['text']).sum()
print(f"Duplicate tweets: {duplicates}")

# Convert timestamp columns to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['collected_at'] = pd.to_datetime(df['collected_at'])

# Extract date and time features
df['date'] = df['timestamp'].dt.date
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()
df['month_day'] = df['timestamp'].dt.strftime('%b %d')

Duplicate tweets: 1


In [31]:
# Clean text for analysis
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove mentions
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags (keep the words)
    text = re.sub(r'#', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [32]:
df['clean_text'] = df['text'].apply(clean_text)

# Handle missing handles
df['handle'] = df['handle'].fillna('Unknown')

print(f"Final dataset shape after cleaning: {df.shape}")


Final dataset shape after cleaning: (145, 10)


In [33]:
df.head()

,name,handle,text,timestamp,collected_at,date,hour,day_of_week,month_day,clean_text
0,Piyush Goyal,@PiyushGoyal,"Rahul Gandhi's statement points towards a major conspiracy by the Congress, along with all oppos...",2026-05-24 04:42:31+00:00,2026-05-25 10:31:04.140113,2026-05-24,4,Sunday,May 24,"Rahul Gandhi's statement points towards a major conspiracy by the Congress, along with all oppos..."
1,India गठबंधन,@IndiaAllliance,"The Modi government will fall within 1 year\n\n- LOP Rahul Gandhi\n\nFor your information, let m...",2026-05-24 03:02:48+00:00,2026-05-25 10:31:04.208680,2026-05-24,3,Sunday,May 24,"The Modi government will fall within 1 year - LOP Rahul Gandhi For your information, let me tell..."
2,Sambit Patra,@sambitswaraj,"Rahul Gandhi said in the Congress Party's minority advisory meeting that, ""In the next one year,...",2026-05-24 05:56:39+00:00,2026-05-25 10:31:04.295123,2026-05-24,5,Sunday,May 24,"Rahul Gandhi said in the Congress Party's minority advisory meeting that, ""In the next one year,..."
3,SriSathya,@sathyashrii,Rahul's Prediction....,2026-05-24 10:53:53+00:00,2026-05-25 10:31:08.276983,2026-05-24,10,Sunday,May 24,Rahul's Prediction....
4,Loustinianós Augustus,@princepoptimus,"The kind of economic collapse that is coming, if it collapses, its army rule not civilian",2026-05-24 09:21:13+00:00,2026-05-25 10:31:08.352152,2026-05-24,9,Sunday,May 24,"The kind of economic collapse that is coming, if it collapses, its army rule not civilian"
